In [23]:
import torch 
import torch.nn as nn
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [24]:
import seaborn as sns
iris = sns.load_dataset('iris')

In [25]:
data = torch.tensor( iris[iris.columns[0:4]].values ).float()

labels = torch.zeros(len(data), dtype=torch.long)

labels[iris.species=='versicolor'] = 1
labels[iris.species=='virinica'] = 2

In [26]:
train_data, test_data, train_labels, test_labels = train_test_split(data, labels, test_size=.2)

train_dataDataset = TensorDataset(train_data, train_labels)
test_dataDataset = TensorDataset(test_data, test_labels)

train_loader = DataLoader(train_dataDataset, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataDataset, batch_size=test_dataDataset.tensors[0].shape[0])

In [27]:
def createANewModel():

    ANNiris = nn.Sequential(
        nn.Linear(4,64),
        nn.ReLU(),
        nn.Linear(64,64),
        nn.ReLU(),
        nn.Linear(64,3)
    )

    lossfun = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(ANNiris.parameters(), lr=.005)

    return ANNiris, lossfun, optimizer

In [28]:
tmpmodel = createANewModel()[0]

print(tmpmodel)

for i in tmpmodel.named_parameters():
    print(i[0], i[1].shape, i[1].numel())

Sequential(
  (0): Linear(in_features=4, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=3, bias=True)
)
0.weight torch.Size([64, 4]) 256
0.bias torch.Size([64]) 64
2.weight torch.Size([64, 64]) 4096
2.bias torch.Size([64]) 64
4.weight torch.Size([3, 64]) 192
4.bias torch.Size([3]) 3


In [29]:
numepochs = 1000

def trainTheModel(L2lambda):

    trainAcc = []
    testAcc = []
    losses = []

    # Counting the total number of weights in the model
    nweights = 0
    for pname, weight in ANNiris.named_parameters():
        if 'bias' not in pname:
            nweights = nweights + weight.numel()

    # Loop over epochs
    for epochi in range(numepochs):

        # Loop over training data batches
        batchAcc = []
        batchLoss = []
        for X, y in train_loader:

            # Forwardpropagation
            yHat = ANNiris(X)
            loss = lossfun(yHat, y)
    
            # Add L1 term
            L1_term = torch.tensor(0.,requires_grad=True)
    
            # Sum up all abs
            for pname, weight in ANNiris.named_parameters():
                if 'bias' not in pname:
                    L1_term = L1_term + torch.sum(torch.abs(weight))
            # Loss
            loss = loss + L1lambda*L1_term/nweights
    
            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    

        trainAcc.append( np.mean(batchAcc) )
        losses.append( np.mean(batchLoss) )

        X, y = next(iter(test_loader))
        predlabels = torch.argmax( ANNiris(X), axis=1 )
        testAcc.append( 100*torch.mean((predlabels == y).float()).item() )

        return trainAcc, testAcc, losses

In [30]:
ANNiris, lossfun, optimizer = createANewModel()
L1lambda = .001
trainAcc, testAcc, losses = trainTheModel(L1lambda)

/home/alexpy/Documents/DL-Course/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/alexpy/Documents/DL-Course/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [31]:
print(trainAcc)

[np.float64(nan)]
